In [16]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import seaborn as sns
import json
from collections import defaultdict
load_dotenv()

kc_routes_path=os.getenv("KC_ROUTES_PATH")
kc_df=pd.read_json(kc_routes_path,orient='index')

kc_df = kc_df.rename(columns={0: 'kc_route'})
leaf_to_id = dict(zip(kc_df['kc_route'], kc_df.index))

def map_kc_routes_to_ids(kc_routes_list):
    # If it's a string and contains '----', treat it as a single path
    if isinstance(kc_routes_list, str):
        # It's a single path string, convert to list with one item
        kc_routes_list = [kc_routes_list]
    elif not isinstance(kc_routes_list, list):
        return kc_routes_list
    
    # Now process each path
    ids = []
    for full_path in kc_routes_list:
        # Split by '----' and get the last part (leaf)
        leaf = full_path.split('----')[-1] if '----' in full_path else full_path
        # Map to ID, or keep original if not found
        ids.append(leaf_to_id.get(leaf, f"NOT FOUND: {leaf}"))
    return ids

questions_path=os.getenv("QUESTIONS_PATH")
t_df=pd.read_json(questions_path)
question_df=t_df.transpose()
question_df=question_df.reset_index()
question_df['kc_ids']=question_df['kc_routes'].apply(map_kc_routes_to_ids)

analysis_df=question_df[['index','analysis','kc_ids']]
analysis_df=analysis_df.rename(columns={'index':'Q_id'})

responses_path=os.getenv("RESPONSES_TRAIN_PATH")
responses_path=os.path.join(responses_path,"train_valid_sequences.csv")

training_set=pd.read_csv(responses_path)

df = training_set.copy()

for col in ['questions', 'concepts', 'responses','timestamps']:
    if isinstance(df[col].iloc[0], str):
        # If stored as comma-separated string
        df[col] = df[col].str.split(',')
    # If it's already a list, keep it

# explode all columns simultaneously
df = df.explode(['questions', 'concepts', 'responses','timestamps'])

# Convert columns to appropriate types
df['questions'] = df['questions'].astype(int)
df['concepts'] = df['concepts'].astype(int)
df['responses'] = df['responses'].astype(int)

# Filter out padding responses (-1)
df = df[df['responses'] != -1]
df=df[df['questions'] != -1]
df=df[df['concepts'] != -1]

df=df.drop(['selectmasks','is_repeat'],axis=1)

#calculate error rates per question
question_stats = df.groupby('questions').agg(
    attempted=('responses', 'count'),
    correct=('responses', lambda x: (x == 1).sum()),
    wrong=('responses', lambda x: (x == 0).sum())
).reset_index()

question_stats['error_rate'] = question_stats['wrong'] / question_stats['attempted']

# Calculate error rates per KC (concept)
kc_stats = df.groupby('concepts').agg(
    attempted=('responses', 'count'),
    correct=('responses', lambda x: (x == 1).sum()),
    wrong=('responses', lambda x: (x == 0).sum())
).reset_index()

kc_stats['error_rate'] = kc_stats['wrong'] / kc_stats['attempted']

#add the metadata to the questions analysis and kc_routes
analysis_metadata=analysis_df.merge(question_stats[['questions', 'attempted', 'correct', 'wrong', 'error_rate']], 
left_on='Q_id', right_on='questions', how='left').drop('questions', axis=1)

question_metadata= question_df.merge(question_stats[['questions', 'attempted', 'correct', 'wrong', 'error_rate']], 
left_on='index', right_on='questions', how='left').drop('questions', axis=1)

kc_metadata = kc_df.reset_index().rename(columns={'index': 'kc_id'})
kc_metadata=kc_metadata.merge(kc_stats[['concepts', 'attempted', 'correct', 'wrong', 'error_rate']],
left_on='kc_id', right_on='concepts', how='left').drop('concepts', axis=1)

print("Analysis Metadata:")
print(analysis_metadata.head())
print("Question Metadata:")
print(question_metadata.head())
print("KC Metadata:")
print(kc_metadata.head())



Analysis Metadata:
   Q_id                                           analysis  kc_ids  attempted  \
0     0  小宇先选，有$$4$$种，接下来小明选，有$$3$$种，最后小丽选，有$$2$$种，所以一共...     [0]     2233.0   
1     1  先分成三类：情况 $$1$$：取的是英语和语文书时有：$$2\times 4=8$$（种）；...  [1, 2]     1472.0   
2     2  先考虑$$A$$区，有$$5$$种选择，接下来$$B$$区有$$4$$种，$$C$$区有$$...     [3]     2170.0   
3     3  第一步：给$$A$$染色，有$$4$$种颜色可选．\n第二步：给$$B$$染色，由于$$B$...     [3]     3233.0   
4     4                         $$3\times 3\times 3=27$$个．     [4]      767.0   

   correct  wrong  error_rate  
0   2023.0  210.0    0.094044  
1    952.0  520.0    0.353261  
2   1783.0  387.0    0.178341  
3   2822.0  411.0    0.127127  
4    686.0   81.0    0.105606  
Question Metadata:
   index                                            content  \
0      0  学校有舞蹈，唱歌、围棋、绘画四种兴趣班． 小宇、小明、小丽三个小朋友准备报名，每人只能报一个...   
1      1  书架上有 $$2$$ 本不同的英语书，$$4$$ 本不同的语文书，$$3$$ 本不同的数学书...   
2      2  用$$5$$种不同的颜色给下面的图形染色，要求相邻的区域（有公共边的两区域称为相邻）染成不同...   
3      3  用四种颜色去涂如图所示的三块区域，要求相邻

In [ ]:
  # Load environment variables from .env file
load_dotenv()

analysis_metadata.to_csv(os.getenv("ANALYSIS_METADATA_PATH"), index=False,encoding='utf-8')
question_metadata.to_csv(os.getenv("QUESTION_METADATA_PATH"), index=False,encoding='utf-8')  
kc_metadata.to_csv(os.getenv("KC_METADATA_PATH"), index=False,encoding='utf-8')

   fold    uid  questions  concepts  responses     timestamps
0     0  11066       3751       187          1  1595229836000
0     0  11066       3752       187          1  1595233013000
0     0  11066       3753       374          1  1595233687000
0     0  11066       3754       187          0  1595236010000
0     0  11066       1990       374          1  1595314541000
